In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append('../../')  # Add project root to path

import logging
from modules.base_ingestion import BaseBronzeIngestion
from typing import List, Dict, Any

# Log Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ProposicoesIngestion(BaseBronzeIngestion):
    """Concrete implementation for Bills ingestion."""
    
    def __init__(self, spark, entity_name: str = 'proposicoes'):
        """Initialize calling base class with only Spark and entity name."""
        super().__init__(
            spark=spark,
            entity_name=entity_name
        )
    
    def fetch_data(self) -> List[Dict[str, Any]]:
        """Implements the abstract method - uses _fetch_standard from base class."""
        logger.info(f"Starting {self.config.entity} ingestion pipeline...")
        
        all_data = []
        params = self.entity_config['params'].copy()
        
        # Extract list of years and iterate over each one
        anos = params.pop('ano') if 'ano' in params else [None]
        
        for ano in anos:
            if ano:
                logger.info(f"Fetching proposicoes for year {ano}...")
                params['ano'] = ano
            else:
                logger.info(f"Fetching proposicoes without year filter...")
            
            data = self._fetch_standard(params=params)
            
            if data:
                all_data.extend(data)
                logger.info(f"Retrieved {len(data)} records for year {ano}")
            else:
                logger.warning(f"No data found for year {ano}")
        
        logger.info(f"Total consolidated: {len(all_data)} records across {len(anos)} year(s)")
        return all_data

# Create ingestion instance
ingestion = ProposicoesIngestion(spark=spark)

In [0]:
try:
    ingestion.execute()
    logger.info(f"{ingestion.entity_name} ingestion completed successfully!")
except Exception as e:
    logger.error(f"{ingestion.entity_name} ingestion failed: {str(e)}")
    raise

In [0]:
%sql
SELECT prop.ano, count(*) 
FROM workspace.camara_bronze.proposicoes as prop
GROUP BY prop.ano
ORDER BY prop.ano